# KrediLens – Keşifsel Analiz (İP 2, Hedef 2.5)

Bu defter **yalnızca eğitim (2007–2013) ve doğrulama (2014)** kümelerini kullanır.
`test.parquet` modelleme bitene kadar açılmaz.

Önce `./run_data_pipeline.sh` çalıştırılmış olmalıdır.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

train = pd.read_parquet("../data/processed/train.parquet")
valid = pd.read_parquet("../data/processed/valid.parquet")
df = pd.concat([train.assign(kume="egitim"), valid.assign(kume="dogrulama")], ignore_index=True)
df["revol_util"] = pd.to_numeric(df["revol_util"].astype(str).str.replace("%", ""), errors="coerce")
df["fico"] = (df["fico_range_low"] + df["fico_range_high"]) / 2
print(df.shape); df.head()

## 1. Sınıf dengesi

In [ ]:
df.groupby("kume")["target"].agg(kayit="size", batik_orani=lambda s: round(s.mean()*100, 2))

## 2. Eksik değer oranları

In [ ]:
miss = df.isna().mean().mul(100).round(2).sort_values(ascending=False)
miss[miss > 0]

## 3. Batık oranı: yıl, FICO, kredi amacı, borç/gelir

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

by_year = df.groupby("issue_year")["target"].mean().mul(100)
by_year.plot.bar(ax=axes[0,0], color="#4C72B0", title="Yıl"); axes[0,0].set_ylabel("Batık oranı (%)")

df["fico_aralik"] = pd.cut(df["fico"], [600, 660, 680, 700, 720, 750, 850], right=False)
df.groupby("fico_aralik", observed=True)["target"].mean().mul(100).plot.bar(ax=axes[0,1], color="#DD8452", title="FICO aralığı")

df.groupby("purpose")["target"].mean().mul(100).sort_values().plot.barh(ax=axes[1,0], color="#55A868", title="Kredi amacı")

df["dti_aralik"] = pd.cut(df["dti"], [0, 10, 15, 20, 25, 30, 100], right=False)
df.groupby("dti_aralik", observed=True)["target"].mean().mul(100).plot.bar(ax=axes[1,1], color="#C44E52", title="Borç/gelir oranı")

plt.tight_layout()

## 4. Sayısal dağılımlar ve uç değerler

In [ ]:
df[["loan_amnt", "annual_inc", "dti", "fico", "open_acc", "revol_util"]].describe(percentiles=[.01, .5, .99]).T

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["loan_amnt", "annual_inc", "dti"]):
    sns.histplot(df[col].clip(upper=df[col].quantile(.99)), bins=40, ax=ax)
    ax.set_title(f"{col} (%99'da kırpılmış)")
plt.tight_layout()

## 5. Metin biçimli sütunlar

In [ ]:
df["emp_length"].value_counts(dropna=False)

## Ön işleme kararları (3. haftaya aktarılacak)

- Sayısal eksikler eğitim kümesinin **medyanı**, kategorik eksikler **"bilinmiyor"** seviyesi ile doldurulacak.
- `emp_length` ve `revol_util` metinden sayıya çevrilecek (`data/variable_dictionary.csv`).
- `annual_inc` logaritma alınıp %99'da kırpılacak.
- Sınıf dengesizliği nedeniyle eğitimde sınıf ağırlığı, değerlendirmede ROC-AUC ve KS kullanılacak.